In [1]:
import fiftyone as fo
import fiftyone.zoo as foz

c:\Users\navee\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
animals = [
    "Fox",
    "Elephant",
    "Leopard",
    "Tiger",
    "Bear",
    "Monkey",       
    "Deer",
]


In [ ]:
all_datasets = []

for cls in animals:
    print(f"Loading dataset for class: {cls}...")
    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        label_types=["detections"],
        classes=[cls],
        max_samples=1000
    )
    print(f"Loaded dataset for class: {cls} with {len(ds)} samples.")
    all_datasets.append(ds)

print("Merging datasets...")

Loading dataset for class: Fox...
Only found 484 (<1000) samples matching your requirements
Necessary images already downloaded
Existing download of split 'train' is sufficient
Loading existing dataset 'open-images-v7-train-1000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use
Loaded dataset for class: Fox with 1000 samples.
Loading dataset for class: Elephant...
Necessary images already downloaded
Existing download of split 'train' is sufficient
Loading existing dataset 'open-images-v7-train-1000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use
Loaded dataset for class: Elephant with 1000 samples.
Loading dataset for class: Leopard...
Only found 694 (<1000) samples matching your requirements
Necessary images already downloaded
Existing download of split 'train' is sufficient
Loading existing dataset 'open-images-v7-train-1000'. To reload from disk, either delete the existing dataset o

In [13]:
i=1
for ds in all_datasets:
    print(i,"-",len(ds))
    i+=1


1 - 1000
2 - 1000
3 - 1000
4 - 1000
5 - 1000
6 - 1000
7 - 1000


In [36]:
print("Merging datasets...")

merged_dataset = fo.Dataset(name="merged-animals")
total_samples = 0

for i, ds in enumerate(all_datasets):
    num_samples = len(ds)
    merged_dataset.add_samples(ds)
    total_samples += num_samples
    print(f"Added {num_samples} samples from class {animals[i]} (total: {total_samples})")

print(f"Merged dataset created with {len(merged_dataset)} samples.")
print(merged_dataset)


Merging datasets...


ValueError: Dataset name 'merged-animals' is not available

In [37]:
import fiftyone.utils.random as four

# Split into train (80%) and val (20%) with single seed
four.random_split(
    merged_dataset, 
    {"train": 0.8, "val": 0.2}, 
    seed=51
)
print("Split complete:")


Split complete:


In [38]:
print("Dataset schema:", merged_dataset.get_field_schema())
print("\nFirst sample fields:")
sample = merged_dataset.first()
print([k for k,v in sample.to_dict().items() if v is not None])

print("\nSample detections:")
print(sample)


Dataset schema: OrderedDict({'id': <fiftyone.core.fields.ObjectIdField object at 0x000001C532B66930>, 'filepath': <fiftyone.core.fields.StringField object at 0x000001C5A0718200>, 'tags': <fiftyone.core.fields.ListField object at 0x000001C532B66A20>, 'metadata': <fiftyone.core.fields.EmbeddedDocumentField object at 0x000001C532B44170>, 'created_at': <fiftyone.core.fields.DateTimeField object at 0x000001C532B66BD0>, 'last_modified_at': <fiftyone.core.fields.DateTimeField object at 0x000001C532B441D0>, 'ground_truth': <fiftyone.core.fields.EmbeddedDocumentField object at 0x000001C532B45100>})

First sample fields:
['filepath', 'tags', 'created_at', 'last_modified_at', 'ground_truth']

Sample detections:
<Sample: {
    'id': '695fc4b27ff6fed10ec06c8c',
    'media_type': 'image',
    'filepath': 'C:\\Users\\navee\\fiftyone\\open-images-v7\\train\\data\\0000f2101250b009.jpg',
    'tags': ['train'],
    'metadata': None,
    'created_at': datetime.datetime(2026, 1, 8, 14, 52, 34, 7000),
    '

In [39]:
train_view = merged_dataset.match_tags("train")
val_view = merged_dataset.match_tags("val")

print(f"Total train: {len(train_view)}, Total val: {len(val_view)}")

train_view.export(
    export_dir="./dataset/train",
    dataset_type=fo.types.COCODetectionDataset,
    label_field="ground_truth",  # Use this field name!
    classes=animals
)

val_view.export(
    export_dir="./dataset/val",
    dataset_type=fo.types.COCODetectionDataset,
    label_field="ground_truth",  # Use this field name!
    classes=animals
)


Total train: 7000, Total val: 1400
Directory './dataset/train' already exists; export will be merged with existing files


  11% |█/-------------|  804/7000 [5.1s elapsed, 37.9s remaining, 210.0 samples/s]    


FileNotFoundError: [WinError 2] The system cannot find the file specified: 'C:\\Users\\navee\\fiftyone\\open-images-v7\\train\\data\\074ef5c9a139dde2.jpg'

In [40]:
train_view = merged_dataset.match_tags("train").exists("detections")
val_view = merged_dataset.match_tags("val").exists("detections")

print(f"Train with detections: {len(train_view)}")
print(f"Val with detections: {len(val_view)}")

# Export safely
train_view.export(
    export_dir="./dataset/train",
    dataset_type=fo.types.COCODetectionDataset,
    label_field="detections",
    classes=animals
)

val_view.export(
    export_dir="./dataset/val",
    dataset_type=fo.types.COCODetectionDataset,
    label_field="detections",
    classes=animals
)


Train with detections: 0
Val with detections: 0
Directory './dataset/train' already exists; export will be merged with existing files


 100% |█████████████████████| 0/0 [14.6ms elapsed, ? remaining, ? samples/s] 
Directory './dataset/val' already exists; export will be merged with existing files
 100% |█████████████████████| 0/0 [10.3ms elapsed, ? remaining, ? samples/s] 
